# 4 · Performance Metrics

Computes Sharpe, Sortino, Calmar, Max Drawdown, and CAPM metrics  
for Equal-Weight and Max-Sharpe portfolios.

**Note:** Beta/Alpha/Treynor here use the equal-weighted asset mean as a proxy market.  
For Nifty-benchmarked figures see nb 6.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
sys.path.insert(0, '..')
from utils import (
    ANALYSIS_DIR, RF_MONTHLY, RF_ANNUAL, PERIODS_PER_YEAR,
    annualise_return, annualise_vol,
    beta, jensen_alpha, treynor_ratio,
    sharpe_ratio, sortino_ratio, max_drawdown, calmar_ratio,
    portfolio_return, portfolio_volatility
)

analysis_path = Path(ANALYSIS_DIR)

returns = pd.read_csv(
    analysis_path / "india_equity_returns.csv",
    index_col="Date", parse_dates=True
)
weights = pd.read_csv(analysis_path / "portfolio_weights.csv")

w_equal = weights["Equal_Weight"].values
w_opt   = weights["MaxSharpe_Weight"].values
w_minvar= weights["MinVar_Weight"].values

port_equal  = returns @ w_equal
port_opt    = returns @ w_opt
port_minvar = returns @ w_minvar
market      = returns.mean(axis=1)      # equal-weighted proxy

def full_metrics(label, port, market, w=None):
    mu = returns.mean().values
    cov = returns.cov().values

    b = beta(port, market)
    sr_monthly = (port.mean() - RF_MONTHLY) / port.std()
    sr_annual  = sr_monthly * np.sqrt(PERIODS_PER_YEAR)

    return {
        "Portfolio"        : label,
        # --- returns ---
        "Monthly_Return"   : round(port.mean(), 6),
        "Annual_Return"    : round(annualise_return(port.mean()), 6),
        "Monthly_Vol"      : round(port.std(), 6),
        "Annual_Vol"       : round(annualise_vol(port.std()), 6),
        # --- risk-adjusted (monthly) ---
        "Sharpe_Monthly"   : round(sr_monthly, 6),
        "Sharpe_Annual"    : round(sr_annual, 6),
        "Sortino_Monthly"  : round(sortino_ratio(port, RF_MONTHLY), 6),
        "Max_Drawdown"     : round(max_drawdown(port), 6),
        "Calmar"           : round(calmar_ratio(port), 6),
        # --- CAPM (proxy market) ---
        "Beta_Proxy"       : round(b, 6),
        "Jensen_Alpha_Mo"  : round(jensen_alpha(port, market, b, RF_MONTHLY), 6),
        "Treynor_Monthly"  : round(treynor_ratio(port, b, RF_MONTHLY), 6),
    }

metrics = pd.DataFrame([
    full_metrics("Equal Weight", port_equal,  market),
    full_metrics("Max Sharpe",   port_opt,    market),
    full_metrics("Min Variance", port_minvar, market),
])

metrics.to_csv(analysis_path / "portfolio_metrics.csv", index=False)

print(f"RF (monthly) : {RF_MONTHLY*100:.4f}%  [{RF_ANNUAL*100:.2f}% p.a. ÷ {PERIODS_PER_YEAR}]")
print("\nPortfolio Metrics:")
print(metrics.to_string(index=False))
print("\nSaved: analysis/portfolio_metrics.csv")